# HashMap (Hash Table)

HashMap is one of the most important data structures in computer science and the backbone of countless LeetCode problems. Understanding how it works internally helps you use it effectively and predict its performance.

---

## Part 1: Theory

### What is a HashMap?

A **HashMap** (also called Hash Table, Dictionary) maps **keys** to **values** using a **hash function**.

```
Key  →  Hash Function  →  Index  →  Value
"cat"  →  h("cat") = 3  →  bucket[3]  →  "meow"
```

In Python, the built-in `dict` is a hash map.

### How It Works Internally

```
HashMap internally = array of "buckets"

  Index:  0       1       2       3       4
         [ ]    [b:2]    [ ]   [a:1]   [c:3]
           ↑               ↑
       empty bucket    key 'a' hashed to index 3
```

**Steps for `map[key] = value`**:
1. Compute `index = hash(key) % array_size`
2. Store `(key, value)` at `array[index]`

**Steps for `map[key]` (lookup)**:
1. Compute `index = hash(key) % array_size`
2. Return value at `array[index]`

### Hash Collisions

Two keys can hash to the same index — this is a **collision**.

**Resolution strategies**:
- **Chaining**: Each bucket holds a linked list of (key, value) pairs
- **Open Addressing**: Find the next empty bucket (linear probing, quadratic probing)

Python's `dict` uses open addressing with a more sophisticated scheme.

```
Chaining example (bucket 2 has collision):
  bucket[2] → [("dog", 4)] → [("god", 7)] → None
```

### Time Complexity

| Operation | Average | Worst Case (many collisions) |
|-----------|---------|------------------------------|
| Insert | O(1) | O(n) |
| Lookup | O(1) | O(n) |
| Delete | O(1) | O(n) |
| Iteration | O(n) | O(n) |

**In practice**: Collisions are rare with a good hash function → effectively O(1).

### Space Complexity

- O(n) where n = number of key-value pairs stored
- Typically 2–3x more memory than the raw data due to empty buckets (load factor)

### Load Factor & Rehashing

- **Load factor** = `entries / buckets`
- When load factor exceeds a threshold (~0.75), the hash map **rehashes**: doubles array size and re-inserts all entries → O(n) but amortized O(1)

### What Can Be a Key?

Keys must be **hashable** — their hash value must not change over their lifetime.

| Hashable (valid keys) | Not Hashable (invalid keys) |
|----------------------|-----------------------------|
| `int`, `float`, `str` | `list` |
| `tuple` (of hashable) | `dict` |
| `frozenset` | `set` |

---

## Part 2: Python HashMap Basics

In [ ]:
# --- Basic Operations ---
d = {}

# Insert / Update
d['a'] = 1
d['b'] = 2
d['a'] = 10  # Update existing key

# Lookup
print(d['a'])           # 10
print(d.get('c', 0))    # 0 (default if not found — safer than d['c'])

# Check existence
print('b' in d)         # True
print('z' in d)         # False

# Delete
del d['b']

# Iterate
d = {'x': 1, 'y': 2, 'z': 3}
for key, val in d.items():
    print(f"{key}: {val}")

In [ ]:
from collections import defaultdict, Counter

# defaultdict — no KeyError on missing key
freq = defaultdict(int)
for ch in "aabbccc":
    freq[ch] += 1
print(dict(freq))   # {'a': 2, 'b': 2, 'c': 3}

# Counter — frequency map in one line
freq2 = Counter("aabbccc")
print(freq2)        # Counter({'c': 3, 'a': 2, 'b': 2})
print(freq2.most_common(2))  # [('c', 3), ('a', 2)]

---

## Part 3: Common HashMap Patterns

### Pattern 1 — Frequency Count
Count occurrences of each element.

In [ ]:
# Count character frequencies
def char_frequency(s):
    freq = {}
    for c in s:
        freq[c] = freq.get(c, 0) + 1
    return freq

print(char_frequency("balloon"))  # {'b':1,'a':1,'l':2,'o':2,'n':1}

### Pattern 2 — Two-Pass Lookup
Build map in first pass, query in second pass.

In [ ]:
# LeetCode 1: Two Sum
def two_sum(nums, target):
    seen = {}  # value -> index
    for i, num in enumerate(nums):
        complement = target - num
        if complement in seen:
            return [seen[complement], i]
        seen[num] = i
    return []

print(two_sum([2, 7, 11, 15], 9))  # [0, 1]

### Pattern 3 — Group by Derived Key
Group elements that share a computed property.

In [ ]:
# LeetCode 49: Group Anagrams
def group_anagrams(strs):
    groups = defaultdict(list)
    for s in strs:
        key = tuple(sorted(s))  # canonical form
        groups[key].append(s)
    return list(groups.values())

print(group_anagrams(["eat","tea","tan","ate","nat","bat"]))

### Pattern 4 — Last Seen Index
Track the most recent position of each element.

In [ ]:
# LeetCode 3: Longest Substring Without Repeating Characters
def length_of_longest_substring(s):
    last_seen = {}
    left = 0
    max_len = 0
    for right, char in enumerate(s):
        if char in last_seen and last_seen[char] >= left:
            left = last_seen[char] + 1
        last_seen[char] = right
        max_len = max(max_len, right - left + 1)
    return max_len

print(length_of_longest_substring("abcabcbb"))  # 3

### Pattern 5 — Prefix Sum + HashMap
Track cumulative sums to find subarrays with a target sum.

In [ ]:
# LeetCode 560: Subarray Sum Equals K
def subarray_sum(nums, k):
    prefix_count = defaultdict(int)
    prefix_count[0] = 1
    prefix_sum = 0
    count = 0
    for num in nums:
        prefix_sum += num
        count += prefix_count[prefix_sum - k]
        prefix_count[prefix_sum] += 1
    return count

print(subarray_sum([1, 1, 1], 2))  # 2

### Pattern 6 — Complement / Pair Lookup
For each element, look up its "partner" in the map.

In [ ]:
# LeetCode 1679: Max Number of K-Sum Pairs
def max_operations(nums, k):
    freq = Counter(nums)
    ops = 0
    for num in freq:
        complement = k - num
        if complement in freq:
            if num == complement:
                ops += freq[num] // 2
            elif num < complement:
                ops += min(freq[num], freq[complement])
    return ops

print(max_operations([1,2,3,4], 5))  # 2

---

## Part 4: High-Frequency LeetCode Problems

### Problem 1 — LeetCode 242: Valid Anagram

In [ ]:
# Two strings are anagrams if they have the same character frequencies
def is_anagram(s, t):
    return Counter(s) == Counter(t)

# Without Counter:
def is_anagram_v2(s, t):
    if len(s) != len(t):
        return False
    freq = {}
    for c in s:
        freq[c] = freq.get(c, 0) + 1
    for c in t:
        if c not in freq or freq[c] == 0:
            return False
        freq[c] -= 1
    return True

print(is_anagram("anagram", "nagaram"))  # True
print(is_anagram("rat", "car"))          # False

### Problem 2 — LeetCode 347: Top K Frequent Elements

In [ ]:
import heapq

def top_k_frequent(nums, k):
    freq = Counter(nums)
    # Use heap to get top k
    return heapq.nlargest(k, freq.keys(), key=freq.get)

print(top_k_frequent([1,1,1,2,2,3], 2))  # [1, 2]

### Problem 3 — LeetCode 383: Ransom Note

In [ ]:
# Can ransomNote be constructed from magazine letters?
def can_construct(ransomNote, magazine):
    available = Counter(magazine)
    for c in ransomNote:
        if available[c] == 0:
            return False
        available[c] -= 1
    return True

print(can_construct("aa", "aab"))  # True
print(can_construct("aa", "ab"))   # False

### Problem 4 — LeetCode 205: Isomorphic Strings

In [ ]:
# Two strings are isomorphic if characters can be mapped 1-to-1
def is_isomorphic(s, t):
    s_to_t = {}
    t_to_s = {}
    for cs, ct in zip(s, t):
        if cs in s_to_t and s_to_t[cs] != ct:
            return False
        if ct in t_to_s and t_to_s[ct] != cs:
            return False
        s_to_t[cs] = ct
        t_to_s[ct] = cs
    return True

print(is_isomorphic("egg", "add"))   # True
print(is_isomorphic("foo", "bar"))   # False
print(is_isomorphic("paper", "title"))  # True

### Problem 5 — LeetCode 128: Longest Consecutive Sequence

In [ ]:
# Find the longest run of consecutive integers — O(n) using a set (hash set)
def longest_consecutive(nums):
    num_set = set(nums)
    max_len = 0

    for num in num_set:
        # Only start counting from the beginning of a sequence
        if (num - 1) not in num_set:
            length = 1
            while (num + length) in num_set:
                length += 1
            max_len = max(max_len, length)

    return max_len

print(longest_consecutive([100,4,200,1,3,2]))  # 4  (1,2,3,4)
print(longest_consecutive([0,3,7,2,5,8,4,6,0,1]))  # 9

### Problem 6 — LeetCode 2260: Minimum Consecutive Cards to Pick Up

In [ ]:
# Find shortest window containing a duplicate value
def minimum_card_pickup(cards):
    last_seen = {}
    min_len = float('inf')
    for i, card in enumerate(cards):
        if card in last_seen:
            min_len = min(min_len, i - last_seen[card] + 1)
        last_seen[card] = i
    return min_len if min_len != float('inf') else -1

print(minimum_card_pickup([3,4,2,3,4,7]))  # 4

### Problem 7 — LeetCode 2342: Max Sum of Pair With Equal Sum of Digits

In [ ]:
# Group numbers by digit sum, find max pair sum per group
def maximum_sum(nums):
    best = {}  # digit_sum -> largest number seen
    result = -1
    for num in nums:
        ds = sum(int(d) for d in str(num))
        if ds in best:
            result = max(result, best[ds] + num)
            best[ds] = max(best[ds], num)
        else:
            best[ds] = num
    return result

print(maximum_sum([18,43,36,13,7]))  # 54

---

## Part 5: Pattern Summary

| Pattern | Description | Key Problems |
|---------|-------------|-------------|
| **Frequency count** | Count occurrences per element | 242, 383, 1189 |
| **Two-pass lookup** | Build map, then query | 1 (Two Sum) |
| **Group by derived key** | Group elements sharing a property | 49 (Group Anagrams) |
| **Last seen index** | Track most recent position | 3, 2260 |
| **Prefix sum + map** | Find subarrays with target sum | 560, 525 |
| **Complement lookup** | Find partner for each element | 1, 1679 |
| **Set membership** | O(1) existence check | 128, 217 |
| **Bidirectional map** | Map both directions simultaneously | 205 |

---

## Related LeetCode Problems

| Problem | Difficulty | Pattern |
|---------|------------|--------|
| [1. Two Sum](https://leetcode.com/problems/two-sum/) | Easy | Complement lookup |
| [49. Group Anagrams](https://leetcode.com/problems/group-anagrams/) | Medium | Group by derived key |
| [128. Longest Consecutive Sequence](https://leetcode.com/problems/longest-consecutive-sequence/) | Medium | Set membership |
| [205. Isomorphic Strings](https://leetcode.com/problems/isomorphic-strings/) | Easy | Bidirectional map |
| [217. Contains Duplicate](https://leetcode.com/problems/contains-duplicate/) | Easy | Set membership |
| [242. Valid Anagram](https://leetcode.com/problems/valid-anagram/) | Easy | Frequency count |
| [347. Top K Frequent Elements](https://leetcode.com/problems/top-k-frequent-elements/) | Medium | Frequency + heap |
| [383. Ransom Note](https://leetcode.com/problems/ransom-note/) | Easy | Frequency count |
| [525. Contiguous Array](https://leetcode.com/problems/contiguous-array/) | Medium | Prefix sum + map |
| [560. Subarray Sum Equals K](https://leetcode.com/problems/subarray-sum-equals-k/) | Medium | Prefix sum + map |
| [2260. Min Consecutive Cards](https://leetcode.com/problems/minimum-consecutive-cards-to-pick-up/) | Medium | Last seen index |
| [2342. Max Sum Equal Digit Sum](https://leetcode.com/problems/max-sum-of-a-pair-with-equal-sum-of-digits/) | Medium | Group by derived key |

---

## Key Takeaways

| Concept | Key Point |
|---------|----------|
| **O(1) average** | Hash maps trade space for speed |
| **Keys must be hashable** | Use tuples instead of lists as keys |
| **`dict.get(k, default)`** | Safer than `dict[k]` — avoids KeyError |
| **`defaultdict`** | Cleaner for frequency maps and grouping |
| **`Counter`** | One-liner frequency map with useful methods |
| **Order preserved** | Python 3.7+ dicts maintain insertion order |
| **When to use** | Anytime you need O(1) lookup, grouping, or counting |